# Semana 13 — Imagens Docker versionadas e Docker Hub

**Roteiro:** revisão rápida de issue/PR/quality gate da Semana 12; foco em construir uma imagem rastreável, publicar após aprovação e preparar promoção/rollback. Publicar imagem é diferente de implantar a aplicação.

## Objetivos

- relacionar issue, PR aprovado, build e artefato;
- usar tags de versão e commit;
- entender Dockerfile multi-stage e `.dockerignore`;
- publicar sem expor credenciais;
- registrar digest e plano de rollback.

## Revisão rápida da Semana 12 — 8 minutos

1. O que conecta issue, branch e PR?
2. O que é necessário para o quality gate bloquear merge?
3. O que acontece se testes falham: publica a imagem?
4. Aprovação humana e check automático são a mesma coisa?

**Gabarito:** referência issue/PR; checks exigidos em ruleset/branch protection; falha deve impedir publicação; revisão humana e automação são controles diferentes.

## Cadeia de entrega

```text
issue → branch → PR → quality gates → build → tag → Docker Hub → deploy
```

Imagem é o artefato; registry armazena; deploy executa. Para FactoryHub podemos publicar imagem da API e do frontend, se separados. Docker Compose local pode passar a referenciar imagens versionadas.

## Tags e digest

- `factoryhub-api:1.3.0`: versão de release;
- `factoryhub-api:sha-a1b2c3d`: commit específico;
- `latest`: ponteiro móvel, insuficiente sozinho para reprodução/rollback.

Registre digest `sha256:...` e commit. Tags podem ser sobrescritas conforme a configuração do registry; prefira tags protegidas/imutáveis quando disponíveis.

## Exemplo — Dockerfile multi-stage

O primeiro estágio prepara wheels; o runtime recebe apenas dependências e aplicação. Adapte módulo, porta e dependências ao projeto; este exemplo pressupõe `app.py` com objeto Flask `app` e `gunicorn` no `requirements.txt`.

```dockerfile
FROM python:3.12-slim AS builder
WORKDIR /build
COPY requirements.txt .
RUN pip wheel --no-cache-dir --wheel-dir /wheels -r requirements.txt

FROM python:3.12-slim AS runtime
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
WORKDIR /app
RUN useradd --create-home --uid 10001 appuser
COPY --from=builder /wheels /wheels
COPY requirements.txt .
RUN pip install --no-cache-dir --no-index --find-links=/wheels -r requirements.txt \
    && rm -rf /wheels
COPY --chown=appuser:appuser . .
USER appuser
EXPOSE 5000
CMD ["gunicorn", "--bind", "0.0.0.0:5000", "app:app"]
```

## Build e validação local

```bash
git rev-parse --short HEAD
docker build -t usuario/factoryhub-api:dev .
docker run --rm -p 5000:5000 --env-file .env.local usuario/factoryhub-api:dev
```

Valide health endpoint, logs, conexão com banco/MQTT e encerramento. `.env.local` é apenas local e não deve ser versionado. Não copie segredo para a imagem.

## `.dockerignore` e publicação

Exemplo inicial (revise se datasets forem necessários):

```text
.git
.venv
venv
__pycache__
.pytest_cache
.env
.env.*
!.env.example
```

Para publicar, use token do Docker Hub com escopo mínimo. Não use `docker login -p TOKEN`; segredo pode ficar no histórico/processos. Não inclua token em Dockerfile, argumentos gravados ou logs.

## Pipeline GitHub Actions — esqueleto

Cadastre `DOCKERHUB_USERNAME` e `DOCKERHUB_TOKEN` como Actions Secrets. Exemplo pedagógico; revise versões das Actions, permissões, triggers e imagem antes de adotar em produção.

```yaml
name: build-and-publish
on:
  push:
    branches: [main]
    tags: ['v*']
permissions:
  contents: read
jobs:
  quality:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: '3.12'}
      - run: pip install -r requirements.txt pytest
      - run: pytest
  publish:
    needs: quality
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: docker/setup-buildx-action@v3
      - uses: docker/login-action@v3
        with:
          username: ${{ secrets.DOCKERHUB_USERNAME }}
          password: ${{ secrets.DOCKERHUB_TOKEN }}
      - uses: docker/metadata-action@v5
        id: meta
        with:
          images: usuario/factoryhub-api
          tags: |
            type=sha,prefix=sha-
            type=semver,pattern={{version}}
      - uses: docker/build-push-action@v6
        with:
          context: .
          push: true
          tags: ${{ steps.meta.outputs.tags }}
```

Job `publish` depende de `quality`: teste falho não publica. Não exponha secrets a workflows de PRs de forks.

## Construir uma vez, promover o mesmo artefato

```text
commit abc123 → imagem sha-abc123 / digest D
                         ├─ homologação
                         └─ produção usa o mesmo digest D
```

Configuração e segredos entram em runtime; não reconstrua a imagem com conteúdo diferente para cada ambiente. Registre versão, digest, commit e ambiente implantado.

## Rollback e atividade

Rollback significa apontar o serviço para a versão/digest previamente validado, verificar health/logs e conferir compatibilidade dos dados. Migração irreversível pode exigir plano próprio.

Em equipe, escreva issue e critérios; desenhe branch/PR/gates/publicação; escolha tags; liste secrets por nome (sem valor); defina como obter digest e escreva rollback. Se não houver registry autorizado, revise o workflow e valide build local sem publicar em conta pessoal.

## Entrega e referências

Entregue Dockerfile revisado, `.dockerignore`, diagrama do pipeline e evidência de build/imagem. Avaliação: rastreabilidade, gate antes de publicar, ausência de segredos, tag reproduzível e rollback verificável.

- Docker, build best practices: <https://docs.docker.com/build/building/best-practices/>
- Docker, multi-stage builds: <https://docs.docker.com/build/building/multi-stage/>
- Docker Hub access tokens: <https://docs.docker.com/security/for-developers/access-tokens/>
- GitHub, publishing Docker images: <https://docs.github.com/en/actions/use-cases-and-examples/publishing-packages/publishing-docker-images>
- GitHub Actions secrets: <https://docs.github.com/en/actions/security-for-github-actions/security-guides/using-secrets-in-github-actions>